[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/magnetic_resonance/magnetic_resonance.ipynb)

# Magnetic Resonance

This example computes magnetic resonance, following driven spin nutation, power-broadened absorption and dispersion spectra, and Hahn spin-echo refocusing in an inhomogeneous field. Spins placed in a static magnetic field precess in the field's plane, from electron spins in solid-state resonance to nuclear spins in spectroscopy and magnetic resonance imaging. A radio-frequency field tuned near the precession frequency tips the spins away from equilibrium, while coupling to the surrounding bath pulls longitudinal magnetization back toward the static field and dephases transverse coherence.

In the algebra of spacetime, an ensemble spin state is modelled as `0.5 * (antiscalar + spin)`, where `spin` is the intrinsic patch `magnet_plane.dual()` of the plane the angular momentum turns in and `one & state` reads the scalar trace. External fields are extrinsic planes of turning that precess the state through `State.commutator(field)`, while reflection across the lab's timelike axis `observer >> planes` isolates the patches that the lab reads as spin so longitudinal and transverse relaxation act on those planes alone. Both the relaxation-precession generator and its short-time exponential are linear maps on states, and composing doubled waiting maps with pulse rotor sandwiches `waiting(turn(waiting(tip)))` assembles an entire spin-echo sequence before any initial state is supplied. A [conventional twin notebook](magnetic_resonance_space.ipynb) runs the same experiments in the algebra of three-dimensional space with the spin represented as a spatial Bloch vector.

In the vector notation of the Bloch equations, precession and relaxation split into an axial cross product with the rotating-frame field vector $\boldsymbol{\omega} = (\Omega, 0, \Delta)$ and separate component decays on the Bloch vector $\mathbf{r} = (r_x, r_y, r_z)$, $\dot{\mathbf{r}} = \boldsymbol{\omega} \times \mathbf{r} - (r_x / T_2,\, r_y / T_2,\, (r_z - 1) / T_1)$, while composing inhomogeneous pulse sequences embeds $(1, \mathbf{r})$ as a four-component column vector under $4 \times 4$ affine Liouville matrices.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Generator, Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from examples.quantum.magnetic_resonance import render
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()                                         # Vector+
Bivector = ga.gatype.bivector()                                     # Bivector+
# A spin is intrinsic, represented by the patch of the plane it turns in.
Spin = ga.gatype.antibivector()                                     # Antibivector-
# A spin state combines half the antiscalar trace with the spin.
State = ga.gatype(Spin + ga.gatype.antiscalar())                    # Antibivector- + Antiscalar-
Rates = ga.gatype((State, State))                                   # State- <- State-
Evolution = ga.gatype((State, State))                               # State- <- State-
Rotor = ga.gatype.rotor()                                           # Rotor+
one = mv.scalar([1])                                                # [] Scalar+
antiscalar = one.dual()                                             # [] Antiscalar-
# Subtracting the trace read by pairing with `one` leaves the spin planes of a state.
planes = State - antiscalar * (one & State)                         # [] State- <- State-
# Lab rest frame: timelike observer vector and turning planes of the static magnet and RF drive.
observer = mv.t                                                     # [] Vector+
magnet_plane = mv.xy                                                # [] Bivector+
drive_plane = mv.yz                                                 # [] Bivector+
# Relaxation times in microseconds, with angular frequencies in radians per microsecond.
t1, t2 = 10.0, 4.0

## Spin states and the rate generator

Two mechanisms govern a spin state in the frame rotating with the drive. Precession under the detuned static field in `magnet_plane` and the radio-frequency drive in `drive_plane` acts through the commutator `State.commutator(field)`. Relaxation acts on the timelike patches that reflection across `observer` leaves invariant, pulling the component invariant under a half turn in `magnet_plane` back toward `magnet_plane.dual()` at rate `1 / t1` while decaying transverse readable components at rate `1 / t2`. Adding precession and relaxation produces a single linear map `rate_map` over `State`.

In [ ]:
# Sample detuning and drive in rad/us, and an initial spin patch.
detuning, drive = 0.7, 1.3
sample_spin = mv.yz.dual() * 0.4 + mv.zx.dual() * -0.2 + mv.xy.dual() * 0.6    # [] Spin-

In [ ]:
def spin_state(spin: Spin) -> State:
    """Forms the spin state `0.5 * (antiscalar + spin)` (`[...] State-`) of `spin`."""
    return 0.5 * (antiscalar + spin)                                            # [...] State-


def rate_map(detuning: np.ndarray, drive: np.ndarray, magnet_plane: Bivector, drive_plane: Bivector,
             observer: Vector, t1: float, t2: float) -> Rates:
    """Combines rotating-frame precession in `magnet_plane` and `drive_plane` with longitudinal `1 / t1`
    and transverse `1 / t2` relaxation seen by `observer` into one generator (`[...] State- <- State-`)."""
    field = magnet_plane * detuning + drive_plane * drive                       # [...] Bivector+
    # Precession under `field` acts as the infinitesimal rotor `(field * (-dt / 2)).exp()`.
    turning = State.commutator(field)                                           # [...] State- <- State-
    # Reflection across `observer` keeps the timelike patches that `observer` reads as spin.
    readable = (planes + (observer >> planes) / observer.squared()) / 2         # [] State- <- State-
    # A half turn in `magnet_plane` keeps the readable component along the magnet's patch.
    along_magnet = (readable + (magnet_plane >> readable) / magnet_plane.scalar_norm_squared()) / 2   # [] State- <- State-
    # Longitudinal spin relaxes toward `magnet_plane.dual()` scaled by the trace; transverse spin decays to zero.
    relaxing = -(along_magnet - magnet_plane.dual() * (one & State)) / t1 - (readable - along_magnet) / t2   # [] State- <- State-
    return turning + relaxing                                                   # [...] State- <- State-


sample_rates = rate_map(detuning, drive, magnet_plane, drive_plane, observer, t1, t2)  # [] State- <- State-
sample_change = sample_rates(spin_state(sample_spin))                           # [] State-
print("rate of change:\n", sample_change.coefficient_table())

## Driven nutation and the steady state

Under a continuous radio-frequency drive, each spin nutates around the effective rotating-frame field while relaxation damps the motion into a stationary state where the generator vanishes. Because `rate_map` preserves the trace and leaves spacelike patches untouched, adding both projections makes the operator invertible so a single linear solve determines `steady_state`. Repeatedly applying the fourth-order propagator `step_evolution` traces each spin's spiral from thermal equilibrium along `magnet_plane.dual()` into its solved stationary point.

In matrix superoperator notation, short-time evolution over $dt$ reads as the matrix exponential $e^{\mathcal{L}\,dt}$, whose fourth-order series truncation matches the classical fourth-order Runge-Kutta step of a time-independent linear equation.

In [ ]:
nutation_detunings = np.array([0.0, 0.5, 1.0])
nutation_drive = 1.0
nutation_dt = 0.02
n_nutation_steps, nutation_stride = 2000, 5

In [ ]:
def steady_state(rates: Rates, observer: Vector) -> State:
    """Solves for the stationary state (`[...] State-`) of `rates` by pinning the trace to one half and
    the spacelike planes unreadable to `observer` to zero."""
    unreadable = (planes - (observer >> planes) / observer.squared()) / 2       # [] State- <- State-
    return (rates + antiscalar * (one & State) + unreadable).solve(antiscalar * 0.5)   # [...] State-


def step_evolution(rates: Rates, dt: float) -> Evolution:
    """Approximates the short-time propagator over `dt` as the fourth-order composition series of
    `rates * dt` (`[...] State- <- State-`)."""
    exponent = rates * dt                                                       # [...] State- <- State-
    term = total = State
    for order in range(1, 5):
        # Each power in the exponential series composes the previous map with `exponent`.
        term = exponent(term) / order                                           # [...] State- <- State-
        total = total + term
    return total                                                                # [...] State- <- State-


driven_rates = rate_map(nutation_detunings, nutation_drive, magnet_plane, drive_plane, observer, t1, t2)  # [3] State- <- State-
steady_states = steady_state(driven_rates, observer)                            # [3] State-
nutation_step = step_evolution(driven_rates, nutation_dt)                       # [3] State- <- State-

In [ ]:
def trajectory(step_map: Evolution, state: State, n_steps: int) -> Generator[State, None, State]:
    """Yields `state` (`[...] State-`) before each of `n_steps` applications of `step_map` and returns
    the state after the last step."""
    for _ in range(n_steps):
        yield state                                                             # [...] State-
        state = step_map(state)
    return state


equilibrium = spin_state(magnet_plane.dual())                                   # [] State-
nutation_start = equilibrium.broadcast_to(nutation_detunings.shape)             # [3] State-
nutation_states = list(trajectory(nutation_step, nutation_start, n_nutation_steps))
save_figure(render.nutation_figure(nutation_states[::nutation_stride], steady_states), "resonance_nutation");

All trajectories start at thermal equilibrium at the top of the Bloch ball (`z`, corresponding to the spin patch `zt` of `xy`) and spiral into their solved stationary dots. On resonance (red) the spin nutates purely in the `yz` plane perpendicular to the `x` drive axis, whereas larger detunings (purple and blue) tilt the precession cone toward `z` and settle closer to equilibrium.

## Power-broadened spectral lines

Steady-state absorption measures the spin component lagging a quarter turn behind the drive, read by pairing with `-zx`, while dispersion measures the in-phase component paired with `yz`. Because `steady_state` reduces stationary balance to a linear solve, evaluating an entire detuning sweep across weak, intermediate, and saturating drives takes one broadcast call.

In the component form of the Bloch equations, the stationary Bloch vector reads $r_z = (1 + \Delta^2 T_2^2) / d$, $-r_y = \Omega T_2 / d$, and $r_x = \Delta \Omega T_2^2 / d$ with $d = 1 + \Delta^2 T_2^2 + \Omega^2 T_1 T_2$, giving a power-broadened half-width $\sqrt{1 + \Omega^2 T_1 T_2} / T_2$.

In [ ]:
n_line_detunings = 241
line_detunings = np.linspace(-3.0, 3.0, n_line_detunings)
line_drives = np.array([0.05, 0.3, 1.0])

In [ ]:
line_rates = rate_map(line_detunings[:, None], line_drives[None, :], magnet_plane, drive_plane, observer, t1, t2)  # [n_line_detunings, 3] State- <- State-
line_states = steady_state(line_rates, observer)                                # [n_line_detunings, 3] State-
save_figure(render.lines_figure(line_detunings, line_drives, line_states), "resonance_lines");

Normalized per unit drive, the weakest drive (red) traces a tall, narrow Lorentzian absorption peak and sharp antisymmetric dispersion wings governed by transverse dephasing alone. Stronger drives (purple and blue) saturate the longitudinal population difference, lowering peak absorption per unit drive and broadening both curves across detuning.

## Spin echo

Static field inhomogeneity gives each spin in a sample a slightly different detuning, causing an ensemble tipped into the transverse plane to fan out and lose its mean pickup-coil signal long before irreversible dephasing acts. Applying a resonant half-turn pulse in `drive_plane` after a delay flips every spin across the drive axis so faster spins trail slower ones and rephase into a Hahn echo at twice the delay. Each short radio-frequency pulse is a rotor `(drive_plane * (-angle / 2)).exp()` acting on `State` by its sandwich.

In NMR notation, static inhomogeneous dephasing cancels at $t = 2\tau$, leaving an echo envelope $e^{-2\tau / T_2}$.

In [ ]:
n_spins, spread, seed = 300, 2.0, 0
delay, echo_dt = 2.5, 0.01
n_echo_steps = 800
frame_stride, frame_duration_ms = 8, 40

In [ ]:
def drive_pulse(drive_plane: Bivector, angle: float) -> Rotor:
    """Forms the rotor (`[] Rotor+`) of a resonant pulse turning spins in `drive_plane` by `angle`."""
    return (drive_plane * (-angle / 2)).exp()                                   # [] Rotor+


def echo_trajectory(step_map: Evolution, drive_plane: Bivector, state: State, n_before: int,
                    n_after: int) -> Iterator[State]:
    """Yields states (`[...] State-`) through a Hahn spin echo: a quarter-turn pulse in `drive_plane`,
    `n_before` free steps, a half-turn refocusing pulse, and `n_after` free steps."""
    tipped = drive_pulse(drive_plane, np.pi / 2) >> state                       # [...] State-
    state = yield from trajectory(step_map, tipped, n_before)
    turned = drive_pulse(drive_plane, np.pi) >> state                           # [...] State-
    yield from trajectory(step_map, turned, n_after)


spin_detunings = np.random.default_rng(seed).normal(0.0, spread, n_spins)
free_rates = rate_map(spin_detunings, 0.0, magnet_plane, drive_plane, observer, t1, t2)  # [n_spins] State- <- State-
echo_start = equilibrium.broadcast_to(spin_detunings.shape)                     # [n_spins] State-
n_delay_steps = round(delay / echo_dt)
echo_stream = echo_trajectory(
    step_evolution(free_rates, echo_dt), drive_plane, echo_start, n_delay_steps, n_echo_steps - n_delay_steps
)
echo_ensemble = stack(
    show_states(
        echo_stream,
        render.echo_frames,
        spin_detunings,
        echo_dt,
        delay,
        t2,
        frame_stride,
        name="resonance_echo",
        duration_ms=frame_duration_ms,
    )
)                                                                               # [n_echo_steps, n_spins] State-

Viewed down the static field axis on the left, spins coloured from negative (blue) to positive (red) detuning fan around the transverse circle after the initial quarter-turn pulse, flip across the horizontal drive axis at the half-turn pulse, and reconverge at twice the pulse delay. On the right, the mean transverse signal (blue) collapses during the initial fan-out and rises back to the dashed `t2` relaxation envelope at the echo peak before dephasing again.

## Pulse sequences as composed maps

Because `step_evolution` is a linear map on `State`, composing a span with itself `span(span)` doubles the waiting interval at each iteration, reaching a thousand time steps in ten compositions. Turning the quarter-turn and half-turn pulse rotors into state maps `pulse_rotor >> State` lets the entire Hahn echo `waiting(turn(waiting(tip)))` and the unrefocused free induction decay `waiting(waiting(tip))` be assembled and averaged across the inhomogeneous ensemble before any initial state is supplied.

In Liouville superoperator notation, the ensemble-averaged echo propagator reads as $\langle e^{\mathcal{L}\tau}\,\mathcal{R}_\pi\,e^{\mathcal{L}\tau}\,\mathcal{R}_{\pi/2}\rangle$, whose transverse echo amplitude decays as $e^{-t / T_2}$ across total elapsed time $t = 2\tau$.

In [ ]:
n_doublings = 11

In [ ]:
def doubled_spans(span: Evolution, n_doublings: int) -> Iterator[Evolution]:
    """Yields `n_doublings` evolution maps (`[...] State- <- State-`), composing each span with itself
    to double the elapsed time."""
    for _ in range(n_doublings):
        yield span                                                              # [...] State- <- State-
        span = span(span)


waiting = stack(list(doubled_spans(step_evolution(free_rates, echo_dt), n_doublings)))  # [n_doublings, n_spins] State- <- State-
times = 2 * echo_dt * 2.0 ** np.arange(n_doublings)
tip = drive_pulse(drive_plane, np.pi / 2) >> State                              # [] State- <- State-
turn = drive_pulse(drive_plane, np.pi) >> State                                 # [] State- <- State-
echo_map = waiting(turn(waiting(tip))).mean(axis=-1)                            # [n_doublings] State- <- State-
decay_map = waiting(waiting(tip)).mean(axis=-1)                                 # [n_doublings] State- <- State-
echo_states = echo_map(equilibrium)                                             # [n_doublings] State-
decay_states = decay_map(equilibrium)                                           # [n_doublings] State-
save_figure(render.decays_figure(times, echo_states, decay_states, t2), "resonance_echo_decay");

Across three decades of total delay, the unrefocused free decay (red) drops to zero within a microsecond as static detuning spreads the ensemble around the equator, whereas the refocused Hahn echo amplitudes (blue) follow the dashed `t2` curve set by irreversible transverse relaxation.

## Boosting the lab and the observer

Carrying the spin, magnet, drive, and relaxing bath together under a Lorentz boost transforms `magnet_plane`, `drive_plane`, and `observer` by the same rotor sandwich, so the stationary state is carried along by that boost and the resonance physics is unchanged. When a boosted observer instead inspects the lab at rest, the rest-frame spin patches no longer contain that observer's timelike vector, splitting the state into a spin part invariant under the observer's reflection and a spacelike remainder that the moving frame does not read as spin.

In [ ]:
# Lorentz boost combining a rapidity along `x` with a rapidity along `y`.
boost = (mv.xt * 0.4).exp() * (mv.yt * -0.3).exp()                              # [] Rotor+

In [ ]:
boosted_rates = rate_map(
    detuning, drive, boost >> magnet_plane, boost >> drive_plane, boost >> observer, t1, t2
)                                                                               # [] State- <- State-
boosted_steady = steady_state(boosted_rates, boost >> observer)                 # [] State-
rest_steady = steady_state(sample_rates, observer)                              # [] State-

# Pulling the rest steady state into the boosted observer's frame splits it into timelike and spacelike patches.
rest_in_boosted_frame = boost << rest_steady                                    # [] State-
readable_part = 0.5 * (rest_in_boosted_frame + (observer >> rest_in_boosted_frame) / observer.squared())  # [] State-
spacelike_part = rest_in_boosted_frame - readable_part                          # [] State-
print("steady state at rest:\n", rest_steady.coefficient_table())
print("boosted lab carried back:\n", (boost << boosted_steady).coefficient_table())
print("readable spin in boosted frame:\n", readable_part.coefficient_table())
print("spacelike part in boosted frame:\n", spacelike_part.coefficient_table())

In [ ]:
# checks
spin_planes = stack((mv.yz, mv.zx, mv.xy))                                      # [3] Bivector+
sample_readings = np.array([0.4, -0.2, 0.6])
expected_rates = np.cross([drive, 0.0, detuning], sample_readings) - np.array(
    [sample_readings[0] / t2, sample_readings[1] / t2, (sample_readings[2] - 1) / t1]
)
assert ((2 * (spin_planes & sample_change.select.antibivector()) - expected_rates).abs() < 1e-12).all()
assert ((nutation_states[-1] - steady_states).antinorm() < 1e-2).all()
assert (driven_rates(steady_states).antinorm() < 1e-12).all()
assert (line_rates(line_states).antinorm() < 1e-12).all()

grid_detunings, grid_drives = line_detunings[:, None], line_drives[None, :]
denominator = 1 + (grid_detunings * t2) ** 2 + grid_drives**2 * t1 * t2
line_spins = 2 * line_states.select.antibivector()                              # [n_line_detunings, 3] Spin-
assert (((mv.xy & line_spins) - (1 + (grid_detunings * t2) ** 2) / denominator).abs() < 1e-12).all()
assert ((-(mv.zx & line_spins) - grid_drives * t2 / denominator).abs() < 1e-12).all()
assert (((mv.yz & line_spins) - grid_detunings * grid_drives * t2**2 / denominator).abs() < 1e-12).all()

refocused_mean = echo_ensemble[2 * n_delay_steps].mean(axis=-1).select.antibivector()  # [] Spin-
refocused_transverse = (refocused_mean - (mv.xy >> refocused_mean)).antinorm()  # [] Scalar+
assert ((refocused_transverse - np.exp(-2 * delay / t2)).abs() < 0.02).all()

echo_spins = echo_states.select.antibivector()                                  # [n_doublings] Spin-
echo_transverse = (echo_spins - (mv.xy >> echo_spins)).antinorm()               # [n_doublings] Scalar+
assert ((echo_transverse - np.exp(-times / t2)).abs() < 1e-5).all()

all_planes = stack((mv.yz, mv.zx, mv.xy, mv.xt, mv.yt, mv.zt))                  # [6] Bivector+
boost_difference = boosted_steady - (boost >> rest_steady)                      # [] State-
assert ((all_planes & boost_difference.select.antibivector()).abs() < 1e-12).all()
assert ((one & boost_difference).abs() < 1e-12).all()
spacelike_planes = stack((mv.xt, mv.yt, mv.zt))                                 # [3] Bivector+
assert ((spacelike_planes & spacelike_part.select.antibivector()).abs() > 0.01).any()